# NR Synchronization Exploration

Notebook for exploring sequences, OFDM, channel effects and synchronization metrics.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from nr_sync.sequences import generate_pss, generate_sss, pci_from_ids

print("NR synchronization modules imported")
print(f"Project root: {PROJECT_ROOT}")

## G2 Noiseless visual validation

This notebook turns the automated gate into report-ready evidence. The checks cover all PSS, SSS and PCI inputs, then the plots show the actual generated vectors and the complete PCI mapping.

In [ ]:
repetitions = 100
pss_sequences = [generate_pss(nid2) for nid2 in range(3)]
selected_nid1 = [0, 100, 200, 335]
sss_matrix = np.array([generate_sss(nid1, 0) for nid1 in selected_nid1])
pci_values = {
    pci_from_ids(nid1, nid2)
    for nid1 in range(336)
    for nid2 in range(3)
}

for _ in range(repetitions):
    for nid2 in range(3):
        assert np.array_equal(generate_pss(nid2), pss_sequences[nid2])
    for nid1 in range(336):
        for nid2 in range(3):
            assert np.array_equal(generate_sss(nid1, nid2), generate_sss(nid1, nid2))

assert len(pss_sequences) == 3
assert len(pci_values) == 1008
print("G2 NOISELESS VALIDATION: 100% PASS")
print(f"PSS: {len(pss_sequences)}/3 | SSS: 1008/1008 | PCI: {len(pci_values)}/1008 | repetitions: {repetitions}")

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
figure.suptitle("G2 Noiseless Gate: sequences and PCI coverage", fontsize=16, fontweight="bold")

for nid2, sequence in enumerate(pss_sequences):
    axes[0, 0].step(np.arange(127), sequence + 2 * nid2, where="mid", label=f"NID2={nid2}")
axes[0, 0].set_title("PSS: all 3 sequences")
axes[0, 0].set_xlabel("Sequence index")
axes[0, 0].set_ylabel("BPSK level + offset")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.25)

image = axes[0, 1].imshow(sss_matrix, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
axes[0, 1].set_title("SSS examples for NID2=0")
axes[0, 1].set_xlabel("Sequence index")
axes[0, 1].set_ylabel("NID1: " + ", ".join(map(str, selected_nid1)))
figure.colorbar(image, ax=axes[0, 1], ticks=(-1, 1), label="BPSK value")

nid1_values = np.arange(336)
for nid2 in range(3):
    axes[1, 0].scatter(nid1_values, 3 * nid1_values + nid2, s=8, label=f"NID2={nid2}")
axes[1, 0].set_title("PCI mapping: 0 to 1007")
axes[1, 0].set_xlabel("NID1")
axes[1, 0].set_ylabel("PCI")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.25)

labels = ["PSS (3)", "SSS (1008)", "PCI (1008)", "Deterministic (100x)"]
axes[1, 1].barh(labels, [1, 1, 1, 1], color="#188977")
axes[1, 1].set_xlim(0, 1.15)
axes[1, 1].set_xticks([0, 1])
axes[1, 1].set_xticklabels(["FAIL", "PASS"])
axes[1, 1].set_title("Validation status")
for index in range(4):
    axes[1, 1].text(1.02, index, "100%", va="center", fontweight="bold")

plt.show()